# Phase 4 - Notebook 06: MVSplat Code Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/06_mvsplat_code_walkthrough.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Navigate the official MVSplat repository structure
2. Understand the model definition and key implementation details
3. Trace the data loading pipeline (RE10K/ACID format)
4. Analyze the training script and hyperparameters
5. Understand the inference pipeline and pretrained model usage

**Estimated Time**: 90 minutes

**Prerequisites**: Notebooks 03 (MVSplat Architecture), 05 (Training & Loss)

**Note**: This notebook analyzes the official MVSplat code without requiring it to be installed. All code snippets are presented as reference.

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import torch
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print("\nThis notebook walks through the official MVSplat code.")
print("GitHub: https://github.com/donydchen/mvsplat")

## 1. Repository Structure

```
mvsplat/
├── src/
│   ├── config/                    # Hydra configuration
│   │   ├── dataset/               # Dataset configs (re10k, acid, dtu)
│   │   ├── model/                 # Model architecture configs
│   │   └── default.yaml           # Default training config
│   ├── dataset/
│   │   ├── dataset_re10k.py       # RealEstate10K data loader
│   │   ├── dataset_acid.py        # ACID data loader  
│   │   ├── dataset_dtu.py         # DTU data loader
│   │   ├── types.py               # Data type definitions
│   │   └── view_sampler/          # View pair sampling strategies
│   │       ├── view_sampler_bounded.py
│   │       └── view_sampler_evaluation.py
│   ├── model/
│   │   ├── encoder/
│   │   │   ├── backbone/          # Feature extraction backbone
│   │   │   │   └── unimatch/      # UniMatch-based encoder
│   │   │   │       ├── backbone.py
│   │   │   │       └── transformer.py
│   │   │   ├── costvolume/        # Cost Volume construction  
│   │   │   │   ├── depth_predictor_multiview.py  # Key file!
│   │   │   │   └── get_depth.py
│   │   │   ├── encoder_mvsplat.py # Main encoder module
│   │   │   └── visualization/
│   │   ├── decoder/
│   │   │   └── decoder_splatting.py  # Gaussian splatting renderer
│   │   └── model_wrapper.py       # PyTorch Lightning wrapper
│   ├── loss/                      # Loss functions
│   ├── scripts/
│   │   ├── train.py               # Training entry point
│   │   └── eval.py                # Evaluation entry point
│   └── misc/                      # Utilities
├── pretrained/                    # Pretrained model weights
└── requirements.txt
```

### Key Files

| File | Role | Our Equivalent |
|------|------|----------------|
| `encoder_mvsplat.py` | Main encoder (backbone + cost vol + heads) | Notebook 03 `SimplifiedMVSplat` |
| `depth_predictor_multiview.py` | Cost Volume + depth prediction | `src/feedforward/cost_volume.py` |
| `decoder_splatting.py` | Gaussian rendering via gsplat | Phase 1 rendering |
| `model_wrapper.py` | Training/eval orchestration | Notebook 05 training loop |
| `dataset_re10k.py` | Data loading | Notebook 05 `SyntheticSceneDataset` |

In [ ]:
# Visualize repository structure

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 11)
ax.axis('off')
ax.set_title('MVSplat Repository Architecture', fontsize=16, fontweight='bold', pad=15)

def draw_box(ax, x, y, w, h, text, color, fontsize=9, subtext=None):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(box)
    dy = 0.12 if subtext else 0
    ax.text(x + w/2, y + h/2 + dy, text,
            ha='center', va='center', fontsize=fontsize, fontweight='bold')
    if subtext:
        ax.text(x + w/2, y + h/2 - 0.18, subtext,
                ha='center', va='center', fontsize=7, style='italic', color='#444')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

# Data layer
draw_box(ax, 0.5, 9.2, 3.5, 1.0, 'Dataset Layer', '#FFE0B2',
         subtext='dataset_re10k.py, view_sampler/')

# Config
draw_box(ax, 5, 9.2, 3, 1.0, 'Config (Hydra)', '#E0E0E0',
         subtext='default.yaml, model/, dataset/')

# Encoder
draw_box(ax, 0.5, 7.0, 7.5, 1.5, 'Encoder (encoder_mvsplat.py)', '#BBDEFB')
draw_box(ax, 0.8, 7.15, 2.0, 0.5, 'Backbone', '#E3F2FD', fontsize=8,
         subtext='UniMatch CNN')
draw_box(ax, 3.1, 7.15, 2.2, 0.5, 'Cost Volume', '#E3F2FD', fontsize=8,
         subtext='depth_predictor_mv')
draw_box(ax, 5.6, 7.15, 2.1, 0.5, 'Gauss Heads', '#E3F2FD', fontsize=8,
         subtext='depth+cov+opacity')

draw_arrow(ax, 2.25, 9.2, 2.25, 8.5)
draw_arrow(ax, 6.5, 9.2, 6.5, 8.5)

# Decoder
draw_box(ax, 0.5, 5.0, 3.5, 1.2, 'Decoder\n(Gaussian Splatting)', '#C8E6C9',
         subtext='gsplat rasterization')
draw_arrow(ax, 4.25, 7.0, 2.25, 6.2)

# Loss
draw_box(ax, 5, 5.0, 3, 1.2, 'Loss Functions', '#F8BBD0',
         subtext='L1 + SSIM + LPIPS')
draw_arrow(ax, 2.25, 5.0, 5, 5.6)

# Wrapper
draw_box(ax, 0.5, 2.8, 7.5, 1.5, 'Model Wrapper (PyTorch Lightning)', '#E1BEE7',
         subtext='training_step, validation_step, configure_optimizers')
draw_arrow(ax, 2.25, 5.0, 2.25, 4.3)
draw_arrow(ax, 6.5, 5.0, 6.5, 4.3)

# Scripts
draw_box(ax, 0.5, 1.0, 3, 1.0, 'train.py', '#FFF9C4', subtext='Training entry')
draw_box(ax, 5, 1.0, 3, 1.0, 'eval.py', '#FFF9C4', subtext='Evaluation entry')
draw_arrow(ax, 2, 2.8, 2, 2.0)
draw_arrow(ax, 6.5, 2.8, 6.5, 2.0)

# Right panel: key details
details = [
    'Key Implementation Details:',
    '',
    '1. Backbone: UniMatch CNN',
    '   - Pretrained feature extractor',
    '   - Outputs multi-scale features',
    '',
    '2. Cost Volume: Cascaded',
    '   - Coarse-to-fine depth planes',
    '   - Multi-scale warping',
    '',
    '3. Renderer: gsplat library',
    '   - CUDA-accelerated splatting',
    '   - Differentiable rasterization',
    '',
    '4. Training: PyTorch Lightning',
    '   - Hydra config management',
    '   - W&B logging',
]

for i, line in enumerate(details):
    weight = 'bold' if i == 0 else 'normal'
    ax.text(9.5, 10 - i * 0.55, line, fontsize=8, fontweight=weight,
            fontfamily='monospace')

plt.tight_layout()
plt.show()

## 2. Model Definition: `encoder_mvsplat.py`

This is the main model file. Let's walk through its key components.

### 2.1 Overview

```python
# Simplified from official src/model/encoder/encoder_mvsplat.py

class EncoderMVSplat(Encoder):
    """MVSplat encoder: backbone + cost volume + Gaussian heads."""
    
    def __init__(self, cfg):
        # 1. Feature backbone (UniMatch CNN)
        self.backbone = CNNEncoder(cfg.backbone)
        
        # 2. Cross-view feature transformer (optional lightweight attention)
        self.cross_view_transformer = CrossViewTransformer(cfg)
        
        # 3. Depth predictor with cost volume
        self.depth_predictor = DepthPredictorMultiView(
            feature_channels=cfg.feature_channels,
            num_depth_candidates=cfg.num_depth_candidates,
            costvolume_unet_feat_dim=cfg.costvolume_unet_feat_dim,
        )
        
        # 4. Gaussian parameter heads
        self.to_gaussians = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.num_gaussians_per_pixel * cfg.d_gaussian),
        )
    
    def forward(self, batch):
        # Extract features
        features = self.backbone(batch['images'])
        
        # Optional cross-view transformer
        features = self.cross_view_transformer(features, batch['cameras'])
        
        # Cost volume + depth prediction
        depths, depth_features = self.depth_predictor(
            features, batch['intrinsics'], batch['extrinsics']
        )
        
        # Predict Gaussian parameters
        raw_gaussians = self.to_gaussians(depth_features)
        gaussians = self._activate_gaussians(raw_gaussians, depths)
        
        return gaussians
```

### 2.2 Key Design Decisions

1. **Multiple Gaussians per pixel**: MVSplat can predict `num_gaussians_per_pixel > 1` (typically 1)
2. **UniMatch backbone**: Pretrained on optical flow, good at matching
3. **Cross-view transformer**: Optional lightweight attention before cost volume
4. **Activation functions**: Same as our Notebook 03 (exp for scales, normalize for quaternions, sigmoid for opacity)

In [ ]:
# Demonstrate the Gaussian activation functions used in MVSplat

import torch.nn.functional as F

def activate_gaussians_mvsplat(raw_params, depths, K, extrinsics):
    """
    Activate raw Gaussian parameters following MVSplat conventions.
    
    The official code splits raw_params into components and applies:
    - Depth offset: depth + sigmoid(raw_offset) * scale
    - Scales: exp(raw_scale) clamped to reasonable range
    - Rotations: normalize to unit quaternion
    - Opacity: sigmoid(raw_opacity)
    - SH coefficients: raw (for view-dependent color)
    """
    B, N = raw_params.shape[:2]
    
    # Split into components (MVSplat packs all params into one tensor)
    # Layout: [depth_offset(1), scale(3), rotation(4), opacity(1), sh(3)]
    idx = 0
    depth_offset = torch.sigmoid(raw_params[..., idx:idx+1])  # [0, 1]
    idx += 1
    
    raw_scale = raw_params[..., idx:idx+3]
    scales = torch.exp(raw_scale.clamp(-10, 5))  # Positive, clamped
    idx += 3
    
    raw_rotation = raw_params[..., idx:idx+4]
    rotations = F.normalize(raw_rotation, dim=-1)  # Unit quaternion
    idx += 4
    
    raw_opacity = raw_params[..., idx:idx+1]
    opacities = torch.sigmoid(raw_opacity)  # [0, 1]
    idx += 1
    
    sh_coeffs = raw_params[..., idx:idx+3]  # RGB (or SH)
    
    return {
        'depth_offset': depth_offset,
        'scales': scales,
        'rotations': rotations,
        'opacities': opacities,
        'sh_coeffs': sh_coeffs,
    }


# Test
torch.manual_seed(42)
raw = torch.randn(1, 100, 12)  # 100 Gaussians, 12 params each

activated = activate_gaussians_mvsplat(raw, None, None, None)

print("MVSplat Gaussian Parameter Activations:")
print(f"{'Parameter':15s} | {'Shape':15s} | {'Range':25s} | Activation")
print("-" * 80)
activations = {
    'depth_offset': 'sigmoid → [0, 1]',
    'scales': 'exp(clamp(-10,5)) → R+',
    'rotations': 'L2 normalize → unit quat',
    'opacities': 'sigmoid → [0, 1]',
    'sh_coeffs': 'raw (no activation)',
}

for key, val in activated.items():
    rng = f'[{val.min():.4f}, {val.max():.4f}]'
    act = activations[key]
    print(f"{key:15s} | {str(list(val.shape)):15s} | {rng:25s} | {act}")

## 3. Cost Volume Implementation

### 3.1 `depth_predictor_multiview.py`

This is the core geometric reasoning module. Key differences from our simplified version:

```python
# Official MVSplat cost volume (simplified pseudocode)

class DepthPredictorMultiView(nn.Module):
    def __init__(self, feature_channels, num_depth_candidates, 
                 costvolume_unet_feat_dim):
        # U-Net for cost volume regularization
        self.costvolume_net = CostVolumeUNet(
            feature_channels, costvolume_unet_feat_dim
        )
        # Depth prediction from regularized cost volume
        self.depth_head = nn.Conv2d(costvolume_unet_feat_dim, 1, 1)
    
    def forward(self, features, intrinsics, extrinsics):
        # 1. Build cost volume via plane sweeping
        cost_volume = self.build_cost_volume(
            features, intrinsics, extrinsics
        )  # [B, C, D, H, W]
        
        # 2. Regularize with 3D U-Net
        # This is more sophisticated than our simple 3D CNN!
        regularized = self.costvolume_net(cost_volume)
        
        # 3. Predict depth via soft argmin
        depth_probs = F.softmax(regularized, dim=2)  # Along depth dim
        depth = (depth_probs * self.depth_planes).sum(dim=2)
        
        # 4. Also return features for Gaussian prediction
        features_2d = regularized.mean(dim=2)  # Collapse depth
        
        return depth, features_2d
```

### 3.2 Cost Volume U-Net

The official implementation uses a proper 3D U-Net with:
- Skip connections (not just plain 3D CNN)
- Multiple resolution levels
- Group normalization

In [ ]:
# Compare our simplified vs official implementation

comparison = {
    'Component': [
        'Backbone', 'Feature dim', 'Cost Volume',
        'Depth planes', 'Regularization', 'Depth prediction',
        'Gaussian params', 'Renderer', 'Training',
    ],
    'Our Simplified': [
        'Simple U-Net', '64', 'Basic plane sweep',
        '32 log-uniform', 'Simple 3D CNN', 'Direct regression',
        '1 per pixel', 'Simple splatting', 'Basic loop',
    ],
    'Official MVSplat': [
        'UniMatch CNN (pretrained)', '128-256', 'Cascaded plane sweep',
        '32-64 log-uniform', '3D U-Net + skip connections', 'Soft argmin',
        '1-4 per pixel', 'gsplat (CUDA)', 'PyTorch Lightning + Hydra',
    ],
}

fig, ax = plt.subplots(1, 1, figsize=(16, 6))
ax.axis('off')

# Create table
col_widths = [0.22, 0.33, 0.45]
headers = ['Component', 'Our Simplified', 'Official MVSplat']
colors_header = ['#E0E0E0', '#BBDEFB', '#C8E6C9']

n_rows = len(comparison['Component'])
cell_height = 0.08
y_start = 0.95

# Draw header
x = 0.02
for j, (header, width, color) in enumerate(zip(headers, col_widths, colors_header)):
    rect = plt.Rectangle((x, y_start), width - 0.01, cell_height,
                          facecolor=color, edgecolor='black', lw=1)
    ax.add_patch(rect)
    ax.text(x + width/2, y_start + cell_height/2, header,
            ha='center', va='center', fontsize=9, fontweight='bold')
    x += width

# Draw rows
for i in range(n_rows):
    y = y_start - (i + 1) * cell_height
    x = 0.02
    row_color = '#FAFAFA' if i % 2 == 0 else '#F0F0F0'
    for j, key in enumerate(headers):
        width = col_widths[j]
        rect = plt.Rectangle((x, y), width - 0.01, cell_height,
                              facecolor=row_color, edgecolor='#CCC', lw=0.5)
        ax.add_patch(rect)
        text = comparison[key][i]
        ax.text(x + width/2, y + cell_height/2, text,
                ha='center', va='center', fontsize=8)
        x += width

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('Simplified vs Official Implementation Comparison',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

## 4. Data Loading Pipeline

### 4.1 RE10K Dataset Format

```
re10k/
├── train/
│   ├── <scene_id>/
│   │   ├── <frame_id>.png        # RGB image
│   │   └── ...
│   └── ...
├── test/
│   └── ...
└── poses/
    ├── <scene_id>.txt             # Camera parameters
    └── ...
```

### 4.2 Camera Parameter Format

Each line in the pose file contains:
```
timestamp fx fy cx cy R00 R01 R02 t0 R10 R11 R12 t1 R20 R21 R22 t2
```

This is a 3x4 matrix `[R | t]` flattened row-major.

In [ ]:
# Demonstrate how MVSplat parses camera parameters

def parse_re10k_camera_line(line):
    """
    Parse a single line from RE10K pose file.
    
    Format: timestamp fx fy cx cy R00 R01 R02 t0 R10 R11 R12 t1 R20 R21 R22 t2
    """
    values = [float(x) for x in line.strip().split()]
    
    timestamp = values[0]
    fx, fy, cx, cy = values[1:5]
    
    # Intrinsics matrix
    K = torch.tensor([
        [fx, 0, cx],
        [0, fy, cy],
        [0,  0,  1],
    ], dtype=torch.float32)
    
    # Extrinsics: 3x4 [R | t]
    extrinsics = torch.tensor(values[5:]).reshape(3, 4).float()
    R = extrinsics[:, :3]
    t = extrinsics[:, 3]
    
    # Convert to 4x4 pose matrix
    pose = torch.eye(4)
    pose[:3, :3] = R
    pose[:3, 3] = t
    
    return {
        'timestamp': timestamp,
        'K': K,
        'pose': pose,
    }


# Example camera line (synthetic)
example_line = "0 500.0 500.0 320.0 240.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0"

parsed = parse_re10k_camera_line(example_line)
print("Parsed RE10K camera parameters:")
print(f"  Timestamp: {parsed['timestamp']}")
print(f"  Intrinsics (K):")
print(f"    fx={parsed['K'][0,0]:.1f}, fy={parsed['K'][1,1]:.1f}")
print(f"    cx={parsed['K'][0,2]:.1f}, cy={parsed['K'][1,2]:.1f}")
print(f"  Pose (world-to-camera):")
print(f"    {parsed['pose']}")

### 4.3 View Sampling Strategy

The official code uses `ViewSamplerBounded` for training:

```python
# Simplified from view_sampler_bounded.py

class ViewSamplerBounded:
    """
    Sample context and target views with bounded overlap.
    
    Key parameters:
    - min_overlap: minimum visual overlap between views (e.g., 0.5)
    - max_overlap: maximum visual overlap (e.g., 0.9)
    - context_gap: frame distance between context views
    - target_gap: frame distance to target views
    """
    
    def sample(self, scene):
        # 1. Randomly select a reference frame
        ref_idx = random.randint(0, len(scene) - 1)
        
        # 2. Find context frame with appropriate overlap
        src_idx = self.find_frame_with_overlap(
            scene, ref_idx, self.min_overlap, self.max_overlap
        )
        
        # 3. Find target frames for supervision
        target_indices = self.sample_target_frames(
            scene, ref_idx, src_idx
        )
        
        return {
            'context': [ref_idx, src_idx],
            'target': target_indices,
        }
```

**Why overlap matters:**
- Too little overlap → no correspondences, cost volume fails
- Too much overlap → trivial matching, model doesn't learn depth
- Sweet spot (50-90%) → challenging but solvable

In [ ]:
# Visualize view sampling

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Simulate a video sequence with camera positions
np.random.seed(42)
n_frames = 50
t_vals = np.linspace(0, 2 * np.pi, n_frames)
cam_x = np.cos(t_vals) * 3 + np.random.randn(n_frames) * 0.1
cam_z = np.sin(t_vals) * 3 + np.random.randn(n_frames) * 0.1

# Scenario 1: Good overlap (moderate baseline)
ax = axes[0]
ax.set_title('Good View Sampling\n(50-90% overlap)', fontsize=11, fontweight='bold')
ax.scatter(cam_x, cam_z, c='lightgray', s=20, alpha=0.5, label='All frames')
ctx = [10, 14]
tgt = [12]
ax.scatter(cam_x[ctx], cam_z[ctx], c='blue', s=100, zorder=5, label='Context')
ax.scatter(cam_x[tgt], cam_z[tgt], c='red', s=100, marker='*', zorder=5, label='Target')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_aspect('equal')

# Scenario 2: Too little overlap (wide baseline)
ax = axes[1]
ax.set_title('Too Wide Baseline\n(<50% overlap)', fontsize=11, fontweight='bold')
ax.scatter(cam_x, cam_z, c='lightgray', s=20, alpha=0.5)
ctx = [5, 30]
tgt = [18]
ax.scatter(cam_x[ctx], cam_z[ctx], c='blue', s=100, zorder=5, label='Context')
ax.scatter(cam_x[tgt], cam_z[tgt], c='red', s=100, marker='*', zorder=5, label='Target')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_aspect('equal')

# Scenario 3: Too much overlap (narrow baseline)
ax = axes[2]
ax.set_title('Too Narrow Baseline\n(>90% overlap)', fontsize=11, fontweight='bold')
ax.scatter(cam_x, cam_z, c='lightgray', s=20, alpha=0.5)
ctx = [20, 21]
tgt = [20]
ax.scatter(cam_x[ctx], cam_z[ctx], c='blue', s=100, zorder=5, label='Context')
ax.scatter(cam_x[tgt], cam_z[tgt], c='red', s=100, marker='*', zorder=5, label='Target')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_aspect('equal')

plt.suptitle('View Sampling: Baseline Controls Difficulty',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Training Script Analysis

### 5.1 Training Configuration (Hydra)

MVSplat uses Hydra for configuration management:

```yaml
# default.yaml (simplified)
model:
  encoder:
    name: mvsplat
    backbone:
      name: unimatch
      num_scales: 2
    d_model: 128
    num_depth_candidates: 32
    costvolume_unet_feat_dim: 64
  decoder:
    name: splatting

dataset:
  name: re10k
  image_shape: [256, 256]
  view_sampler:
    name: bounded
    min_overlap: 0.5
    max_overlap: 0.9

training:
  lr: 1.5e-4
  weight_decay: 0.05
  warmup_steps: 2000
  max_steps: 300000
  batch_size: 14  # Per GPU
  
loss:
  mse_weight: 1.0
  lpips_weight: 0.05
```

### 5.2 Training Step (PyTorch Lightning)

```python
# From model_wrapper.py (simplified)

class ModelWrapper(LightningModule):
    def training_step(self, batch, batch_idx):
        # 1. Encode: images → Gaussians
        gaussians = self.encoder(batch)
        
        # 2. Decode: Gaussians → rendered images
        rendered = self.decoder(
            gaussians, batch['target']['extrinsics'],
            batch['target']['intrinsics']
        )
        
        # 3. Loss: rendered vs ground truth
        loss = self.loss_fn(
            rendered['color'], batch['target']['image']
        )
        
        return loss
    
    def configure_optimizers(self):
        optimizer = AdamW(
            self.parameters(),
            lr=1.5e-4,
            weight_decay=0.05,
        )
        scheduler = LinearWarmupCosineDecay(
            optimizer,
            warmup_steps=2000,
            max_steps=300000,
        )
        return [optimizer], [scheduler]
```

In [ ]:
# Official training hyperparameters

official_config = {
    'Training': {
        'Learning rate': '1.5e-4',
        'Optimizer': 'AdamW (weight_decay=0.05)',
        'Scheduler': 'Linear warmup (2K) + cosine decay',
        'Max steps': '300,000',
        'Batch size': '14 per GPU (8 GPUs = 112 total)',
        'Gradient clipping': 'max_norm=0.5',
        'Mixed precision': 'FP16 (except depth)',
    },
    'Architecture': {
        'Backbone': 'UniMatch CNN (pretrained)',
        'Feature dim': '128',
        'Depth planes': '32 (log-uniform)',
        'Gaussians per pixel': '1',
        'Image resolution': '256 x 256',
    },
    'Loss': {
        'MSE weight': '1.0',
        'LPIPS weight': '0.05',
        'SSIM weight': '0.0 (not used)',
    },
    'Data': {
        'Dataset': 'RE10K (train) + ACID (train)',
        'Context views': '2',
        'Target views': '1',
        'View overlap': '50-90%',
    },
}

print("Official MVSplat Training Configuration")
print("=" * 55)
for section, params in official_config.items():
    print(f"\n{section}:")
    for key, val in params.items():
        print(f"  {key:25s}: {val}")

## 6. Inference Pipeline

### 6.1 Running MVSplat Inference

```bash
# Clone and setup
git clone https://github.com/donydchen/mvsplat.git
cd mvsplat
pip install -r requirements.txt

# Download pretrained model
# Available at: https://huggingface.co/donydchen/mvsplat
mkdir -p pretrained
wget -O pretrained/re10k.ckpt <url>

# Run inference on RE10K test set
python -m src.scripts.eval \
    model=mvsplat \
    dataset=re10k \
    checkpoint_path=pretrained/re10k.ckpt
```

### 6.2 Inference Data Flow

```
2 input images + camera params
        ↓
    Preprocess (resize, normalize)
        ↓
    Encoder forward pass (~20ms)
        ↓
    Pixel-aligned Gaussians (H×W per view)
        ↓
    Merge views → 2×H×W Gaussians
        ↓
    Render from novel viewpoint (~5ms)
        ↓
    Output image

Total: ~25ms per novel view (~40 FPS)
```

In [ ]:
# Simulate the inference pipeline steps

def simulate_inference_timing():
    """Simulate MVSplat inference timing breakdown."""
    timings = {
        'Preprocessing': 2.0,
        'Feature extraction': 5.0,
        'Cost volume build': 6.0,
        'Cost volume process': 4.0,
        'Gaussian prediction': 2.0,
        'Back-projection': 1.0,
        'Rendering (gsplat)': 5.0,
    }
    return timings


timings = simulate_inference_timing()
total = sum(timings.values())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
ax = axes[0]
names = list(timings.keys())
values = list(timings.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))
bars = ax.barh(names, values, color=colors, edgecolor='black', lw=0.5)
ax.set_xlabel('Time (ms)', fontsize=11)
ax.set_title(f'Inference Timing Breakdown\n(Total: {total:.0f}ms = {1000/total:.0f} FPS)',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}ms', va='center', fontsize=9)
ax.grid(True, alpha=0.3, axis='x')

# Pie chart
ax = axes[1]
ax.pie(values, labels=names, autopct='%1.0f%%', colors=colors,
       textprops={'fontsize': 8}, pctdistance=0.8)
ax.set_title('Time Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Total inference time: {total:.0f}ms ({1000/total:.0f} FPS)")
print("Bottleneck: Cost volume build + process (~40% of time)")

## 7. Tips for Working with the Official Code

### 7.1 Common Modifications

| Goal | What to Modify |
|------|---------------|
| Change backbone | `src/model/encoder/backbone/` |
| Adjust depth range | Config `num_depth_candidates`, `min_depth`, `max_depth` |
| Custom dataset | Create new `dataset_*.py` following RE10K format |
| Different loss | Modify `src/loss/` |
| Change # Gaussians/pixel | Config `num_gaussians_per_pixel` |
| Add depth supervision | Add depth loss in `model_wrapper.py` |

### 7.2 Debugging Tips

1. **Visualize cost volume slices** to check if warping is correct
2. **Monitor depth predictions** during training (they should become sharper)
3. **Check Gaussian scales** - if too large, rendering will be blurry
4. **Opacity distribution** - should be bimodal (near 0 or near 1)
5. **Use `overfit_to_scene=True`** to verify the pipeline works on a single scene

In [ ]:
# Summary

summary = """
=====================================================================
   Notebook 06 Summary: MVSplat Code Walkthrough
=====================================================================

1. REPOSITORY STRUCTURE
   - src/model/encoder/ : Main model (backbone + cost vol + heads)
   - src/model/decoder/ : Gaussian splatting renderer (gsplat)
   - src/dataset/       : Data loaders (RE10K, ACID, DTU)
   - src/config/        : Hydra configuration files

2. KEY IMPLEMENTATION DETAILS
   - UniMatch CNN backbone (pretrained on optical flow)
   - 3D U-Net for cost volume regularization
   - Soft argmin for depth prediction
   - gsplat for differentiable rendering

3. DATA PIPELINE
   - RE10K format: images + pose files
   - View sampling: 50-90% overlap between context views
   - 2 context views → 1 target view per training step

4. TRAINING
   - PyTorch Lightning + Hydra config
   - AdamW optimizer, lr=1.5e-4, 300K steps
   - MSE + LPIPS loss (no SSIM in official code)
   - 8 GPUs, batch_size=14 per GPU

5. INFERENCE
   - ~25ms per novel view (~40 FPS)
   - Pretrained models on HuggingFace
   - Bottleneck: cost volume construction

=====================================================================
"""
print(summary)

## What's Next?

**[07_inference_evaluation.ipynb](./07_inference_evaluation.ipynb)** - Hands-on inference and evaluation: running pretrained models and computing metrics.

---

## References

1. MVSplat Code: https://github.com/donydchen/mvsplat
2. MVSplat Paper: https://arxiv.org/abs/2403.14627
3. MVSplat Pretrained: https://huggingface.co/donydchen/mvsplat
4. UniMatch: https://arxiv.org/abs/2211.05783
5. gsplat: https://docs.gsplat.studio/